In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input



X_train = np.load("../data/X_train.npy")
y_train = np.load("../data/y_train.npy")
X_val = np.load("../data/X_val.npy")
y_val = np.load("../data/y_val.npy")
X_test = np.load("../data/X_test.npy")
y_test = np.load("../data/y_test.npy")


scaler_features = joblib.load('../models/scaler_features.joblib')
scaler_target = joblib.load('../models/scaler_target.joblib')

print("libraries and data loaded succesfully")

libraries and data loaded succesfully


In [66]:
# 2. Build the LSTM Model
model = Sequential([
    # Input layer: telling the model the shape of our 7-day windows
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    
    # First LSTM layer: 50 units. return_sequences=False because the next layer is Dense
    LSTM(50, activation='relu', return_sequences=False),
    
    # Dropout: randomly turns off 20% of neurons to prevent "memorization" (overfitting)
    Dropout(0.2),
    
    # Output layer: 1 single neuron (the predicted Max Temp)
    Dense(1)
])

# 3. Compile the model
# optimizer='adam' is the industry standard (it's efficient)
# loss='mse' (Mean Squared Error) measures the distance between prediction and reality
model.compile(optimizer='adam', loss='mse')

model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_8 (LSTM)                   │ (None, 50)             │        13,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,051 (50.98 KB)

 Trainable params: 13,051 (50.98 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 4. Train the model
print("Starting training...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1 # This shows the progress bar
)

Starting training...
Epoch 1/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0138 - val_loss: 0.0099
Epoch 2/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0069 - val_loss: 0.0103
Epoch 3/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0062 - val_loss: 0.0082
Epoch 4/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0058 - val_loss: 0.0075
Epoch 5/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0055 - val_loss: 0.0085
Epoch 6/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0055 - val_loss: 0.0077
Epoch 7/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0053 - val_loss: 0.0099
Epoch 8/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0051 - val_loss: 0.0085
Epoch 9/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0050 - val_loss: 0.0082
Epoch 10/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0048 - val_loss: 0.0079
Epoch 11/60
311/311 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0049 - val_loss: 0.0076
Epoch 12/60
311/311 ━━━━━━━━━━━

In [68]:
# 1. Fazer previsões com o Test Set (Dados de 2024)
y_pred_scaled = model.predict(X_test)

# 2. Inverter o escalonamento (Back to Celsius)
# Nota: Precisamos de usar o 'scaler_target' que criaste no notebook 02.
# Se estivesses num novo ficheiro, terias de carregar o scaler gravado.
y_pred = scaler_target.inverse_transform(y_pred_scaled)
y_real = scaler_target.inverse_transform(y_test)

print("Previsões concluídas e convertidas para Celsius.")

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Previsões concluídas e convertidas para Celsius.


In [69]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_real, y_pred)
rmse = np.sqrt(mean_squared_error(y_real, y_pred))

print(f"Erro Médio Absoluto (MAE): {mae:.2f}°C")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f}°C")

Erro Médio Absoluto (MAE): 2.09°C
Raiz do Erro Quadrático Médio (RMSE): 2.77°C


In [70]:
# Criar pasta models se não existir
import os
if not os.path.exists('../models'):
    os.makedirs('../models')

# Guardar o modelo
model.save('../models/weather_predictor_LSTM.keras')
print("Modelo guardado com sucesso em '../models/weather_predictor_LSTM.keras'")

Modelo guardado com sucesso em '../models/weather_predictor_LSTM.keras'
